# Delta Demo — Day 3: Update, Delete, Insert (Deletion Vectors)
**Prerequisite:** Delta_Demo_Day1_Load and Delta_Demo_Day2_Load must already have been run — this notebook assumes the `employees` table already has Day 1 + Day 2's 5 rows.

Runs 4 manual DML statements (UPDATE, DELETE, UPDATE, INSERT), verifies each one's effect on the raw JSON/Parquet files (including how Deletion Vectors work), checks `DESCRIBE HISTORY` for any automatic OPTIMIZE commits Databricks added on its own, and finishes with a small investigation into when Delta actually writes its first checkpoint file.

## Day 3 — Update, Delete, Insert (Deletion Vectors in Action)
This is where things get more advanced. This table has **Deletion Vectors** enabled (a modern Delta feature). That means UPDATE and DELETE do NOT rewrite whole Parquet files the way older "classic" Delta behavior would. Instead, Delta marks specific row *positions* as logically deleted using a small separate file, and leaves the original Parquet file physically untouched. This is called **merge-on-read**. We'll prove this by reading the raw files ourselves right after the UPDATE below.

In [0]:
%sql
-- This UPDATE changes eno=3's name and salary.
-- Because Deletion Vectors are enabled, Delta will NOT rewrite the whole file that
-- eno=3 currently lives in. Watch what actually happens in the next section.
UPDATE delta.`/Volumes/workspace/delta_demo/demo_files/employees`
SET ename = 'Uma Maheshwar', sal = 36999
WHERE eno = 3;

## Day 3 — Raw Contents Deep Dive (Deletion Vector in Action)
This commit is more complex than Day 1/2 — it contains THREE actions instead of just one `add`. Look for:
1. A `remove` action — the ORIGINAL Day-1 file being logically taken out of the current snapshot.
2. An `add` action for a brand NEW small file — holding just the updated row's new values.
3. Another `add` action for the SAME original file, re-added — but this time it carries a `deletionVector` block. This is the mechanism: the old file comes back into the snapshot, just with one specific row masked out.

In [0]:
%sh
cat /Volumes/workspace/delta_demo/demo_files/employees/_delta_log/00000000000000000002.json \
  | while IFS= read -r line; do echo "$line" | python3 -m json.tool; echo "---"; done

In [0]:
%python
# Read every physical Parquet file again. IMPORTANT: because pandas bypasses Delta
# entirely, it will show you eno=3's OLD row (Uma / 35000) still sitting in the
# original file — even though Delta itself will only ever show you the NEW values
# when you query the table normally. This gap between 'what pandas sees on disk'
# and 'what Delta returns to a query' IS what merge-on-read means in practice.
import pandas as pd, glob
for f in sorted(glob.glob("/Volumes/workspace/delta_demo/demo_files/employees/*.parquet")):
    print(f"\n=== {f.split('/')[-1]} ===")
    print(pd.read_parquet(f))

In [0]:
%python
# BONUS: a quick, readable summary of every commit so far — instead of scrolling
# through full raw JSON every time, this just prints which file(s) got added or
# removed by each version, and which operation caused it.
import json, glob

for f in sorted(glob.glob("/Volumes/workspace/delta_demo/demo_files/employees/_delta_log/*.json")):
    print(f"\n=== {f.split('/')[-1]} ===")
    with open(f) as fh:
        for line in fh:
            action = json.loads(line)
            if "add" in action:
                print(f"  ADD    -> {action['add']['path']}  ({action['add']['size']} bytes)")
            elif "remove" in action:
                print(f"  REMOVE -> {action['remove']['path']}")
            elif "commitInfo" in action:
                print(f"  COMMIT -> {action['commitInfo'].get('operation')}")

## Verify the UPDATE Result
Query through Delta — you should see eno=3's NEW values (Uma Maheshwar / 36999), proving Delta is correctly hiding the old row via the deletion vector, even though we just saw the old row still physically exists on disk.

In [0]:
%sql
SELECT * FROM delta.`/Volumes/workspace/delta_demo/demo_files/employees` ORDER BY eno;

## Delete Row (eno = 4)
A DELETE also uses a deletion vector — it marks the row as gone without rewriting the file it lives in.

In [0]:
%sql
DELETE FROM delta.`/Volumes/workspace/delta_demo/demo_files/employees` WHERE eno = 4;

## Update Salary (eno = 5)
One more UPDATE, same deletion-vector mechanism as before.

In [0]:
%sql
UPDATE delta.`/Volumes/workspace/delta_demo/demo_files/employees` SET sal = 50000 WHERE eno = 5;

## Insert New Row (eno = 6)
A plain INSERT again — no deletion vector involved, since nothing is being removed.

In [0]:
%sql
INSERT INTO delta.`/Volumes/workspace/delta_demo/demo_files/employees` VALUES (6, 'Shanmukh J', 41999);

## Describe History — End of Day 3 (Watch for Auto-OPTIMIZE)
We ran exactly 4 statements today (UPDATE, DELETE, UPDATE, INSERT). Look closely at the `operation` column below — you will likely see MORE than 4 new rows. Databricks automatically runs `OPTIMIZE` in the background (visible as `"auto":"true"` in `operationParameters`) to compact small files and clean up deletion vectors. This is NOT something we asked for — it's a serverless default. The transaction log records what Delta does FOR you, not just what you explicitly ran.

In [0]:
%sql
DESCRIBE HISTORY delta.`/Volumes/workspace/delta_demo/demo_files/employees`;

## Checkpoint Verification — At Which Version Does `_last_checkpoint` Appear?
Delta periodically writes a **checkpoint** — a compacted snapshot of the whole table state, so readers don't have to replay every single JSON commit from version 0 every time. The commonly cited default is "every 10 commits," but let's verify that's actually true in THIS environment instead of assuming it.

### Baseline — Confirm No Checkpoint Exists Yet (End of Day 3, Version 8)

In [0]:
%sh
ls -la /Volumes/workspace/delta_demo/demo_files/employees/_delta_log/

### Push the Table to Version 9 (Harmless No-Op Update)

In [0]:
%sql
-- Setting a column to itself still counts as a real transaction/commit — this just
-- gives us one more version number to check, without changing any actual data.
UPDATE delta.`/Volumes/workspace/delta_demo/demo_files/employees` SET sal = sal WHERE eno = 1;

### Check for a Checkpoint File at Version 9

In [0]:
%sh
ls -la /Volumes/workspace/delta_demo/demo_files/employees/_delta_log/

### If No Checkpoint Yet — Push to Version 10 (Harmless No-Op Update)

In [0]:
%sql
UPDATE delta.`/Volumes/workspace/delta_demo/demo_files/employees` SET sal = sal WHERE eno = 2;

### Check for a Checkpoint File at Version 10

In [0]:
%sh
ls -la /Volumes/workspace/delta_demo/demo_files/employees/_delta_log/

### Read `_last_checkpoint` Directly — Definitive Answer

In [0]:
%sh
# If this file doesn't exist, the shell will show an error — that itself IS the
# answer (no checkpoint written yet). If it exists, its contents tell you exactly
# which version the checkpoint was taken at.
cat /Volumes/workspace/delta_demo/demo_files/employees/_last_checkpoint

## Day 3 Summary — List All Objects
Full listing after Day 3. Note: this will show MORE Parquet/deletion-vector files physically on disk than are actually "active" in the current snapshot — files superseded by auto-OPTIMIZE stay on disk until `VACUUM` runs (covered in a later video).

In [0]:
%sh
echo "=== _delta_log/ ==="
ls -la /Volumes/workspace/delta_demo/demo_files/employees/_delta_log/
echo ""
echo "=== data files (Parquet + deletion vectors) ==="
ls -la /Volumes/workspace/delta_demo/demo_files/employees/ | grep -v _delta_log

In [0]:
%sql
SELECT * FROM delta.`/Volumes/workspace/delta_demo/demo_files/employees` ORDER BY eno;